# STAR-RIS RSMA — v20: Scalability theo N (Chứng minh lợi thế phân rã MADDPG)
## Notebook bổ trợ — củng cố §4.7 & Bảng 4.6 ("Khả năng mở rộng: Cao") của luận văn

---

### 📋 Cập nhật v20 (so với lần chạy 3-seed trước)
> Lần chạy 3-seed {16,32,64,96} đã cho **xu hướng đúng**: gain = SR(MADDPG)−SR(TD3) tăng **−0.19 → +0.98** b/s/Hz,
> TD3 còn **suy giảm** ở N=96. v19 **siết chặt thống kê + kéo dài xu hướng**:
>
> | | Lần trước | **v19** |
> |---|---|---|
> | Seeds | 3 | **10** [1000–10000] → CI rất hẹp, thuyết phục tối đa |
> | N | {16,32,64,96} | **{16,32,64,96,128}** → xu hướng càng rõ |
> | Config λ/residual | v18 | **giữ nguyên v18** (công bằng) |
> | **Resume** | không | ✅ **bỏ qua (N,algo) đã chạy** → chạy được qua NHIỀU session |

### 🎯 Kết quả MONG ĐỢI
1. **`scal_sumrate_vs_N.png`** — "scissors plot": 2 đường giao ở N≈20, MADDPG tách top dần; TD3 chững/giảm ở N≥96.
2. **`scal_decomposition_gain.png`** ⭐ — `gain(N)` **đi lên đơn điệu** tới N=128.
3. **`scal_qos_vs_N.png`** — MADDPG vượt QoS dần theo N.
4. CI của TD3 hẹp lại so với 3-seed (nhờ 5 seeds) — phản bác lo ngại "do nhiễu".

### ⚙️ BẮT BUỘC
- **Session Kaggle RIÊNG**, cùng project dataset như v18 (train.py có freeze λ), sửa `PROJECT_ROOT`.

### ⏱️ Compute — 10 seeds × 5 N (gồm N=128) ≈ **40h** → BẮT BUỘC chạy NHIỀU session (Resume)!
> **Hai cách xử lý (cell config sẽ in ước lượng cụ thể):**
> - **Cách A (khuyến nghị) — chạy NHIỀU session nhờ Resume:**
>   1. Session 1: Run All. Khi gần hết giờ, nó đã lưu dần `scalability_vs_N.csv` cho từng (N,algo) xong.
>   2. **Save Version** → ở session 2, thêm output session 1 làm Input, đặt `RESUME_CSV` trỏ tới file đó.
>   3. Run All lại → notebook **bỏ qua (N,algo) đã xong**, chạy tiếp phần còn lại.
> - **Cách B — gọn 1 session:** giảm còn `N_LIST=[16,32,64,96]` **hoặc** `SCAL_SEEDS` 3 seeds (sửa ở cell config).


### 1. Kiểm tra phần cứng & cài thư viện

In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
!pip install gymnasium pyyaml tqdm pandas matplotlib tensorboard

### 2. Đường dẫn dự án & import

In [ ]:
import os, sys, copy, time
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import Image, display

# ⚠️ Sửa cho khớp tên dataset bạn upload lên Kaggle:
PROJECT_ROOT = "/kaggle/input/datasets/duythanhb1909984/star-ris-rsma-maddpg-v14"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from experiments.train import train_maddpg, train_single_agent
from experiments.evaluate import _eval_multi_seed
from utils.plotting import plot_metric_vs_x
from utils import confidence_interval
print("Import OK")

### 3. Cấu hình v18 + tham số scalability

**📌 Output mong đợi (in từ cell dưới):**
- Xác nhận config v18: `freeze=0.55, λ_max=13, target=0.50, residual_scale=0.25`.
- `N_LIST = [16, 32, 64]`, `Algos = ['maddpg','td3']`, `Seeds = [1000,2000,3000]`, `Episodes = 1500`.
- `Số lần train = 18` và **ước lượng thời gian (giờ)** — kiểm tra < 11h trước khi Run All.

> Chỉnh `N_LIST` / `SCAL_SEEDS` / `SCAL_EPISODES` / `SCAL_ALGOS` ở cell code tùy ngân sách. Thêm `"ddpg"` hoặc N=96 nếu còn giờ.


In [ ]:
config_path = os.path.join(PROJECT_ROOT, "config", "config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# ----- Config v18 (giữ nguyên để công bằng) -----
cfg["env"]["qos_lambda_freeze_fraction"] = 0.55
cfg["env"]["qos_lambda_max"] = 13.0
cfg["env"]["qos_target_satisfaction"] = 0.50
cfg["env"]["qos_lambda_increase"] = 1.02
cfg["env"]["phase_residual_scale"] = 0.25

# ----- Tham số scalability (điều chỉnh theo ngân sách) -----
N_LIST        = [16, 32, 64, 96, 128]     # quét N; bỏ 128 nếu muốn gọn 1 session
SCAL_ALGOS    = ["maddpg", "td3"]
SCAL_SEEDS    = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]   # v20: 10 seeds
SCAL_EPISODES = 1500
LABEL = {"maddpg": "MADDPG", "td3": "TD3", "ddpg": "DDPG"}

out_root = "/kaggle/working/"
fig_dir  = os.path.join(out_root, "figures")
tab_dir  = os.path.join(out_root, "tables")
log_dir  = os.path.join(out_root, "logs_scal")
ckpt_dir = os.path.join(out_root, "checkpoints_scal")
for d in (fig_dir, tab_dir):
    os.makedirs(d, exist_ok=True)

CSV_PATH   = os.path.join(tab_dir, "scalability_vs_N.csv")
# RESUME: để chạy tiếp ở session sau, thêm output session trước làm Input rồi trỏ vào đây.
# Vd: RESUME_CSV = "/kaggle/input/scal-part1/scalability_vs_N.csv"
RESUME_CSV = CSV_PATH

# Ước lượng thời gian (thô, GPU T4; anchor: TD3 ~12 phút/seed @ N=96,1500ep)
_td3_n96 = 12.0
est_min = 0.0
for N in N_LIST:
    td3_n  = _td3_n96 * (N / 96.0)        # ~tuyến tính theo N
    cost = {"maddpg": 2.6 * td3_n, "td3": td3_n, "ddpg": 1.0 * td3_n}
    for a in SCAL_ALGOS:
        est_min += cost[a] * len(SCAL_SEEDS)

print("=" * 60)
print("  SCALABILITY SWEEP v19 — cấu hình")
print("=" * 60)
print(f"  N_LIST          = {N_LIST}")
print(f"  Algos / Seeds   = {SCAL_ALGOS} / {SCAL_SEEDS}")
print(f"  Episodes/seed   = {SCAL_EPISODES}")
print(f"  Số lần train    = {len(N_LIST)*len(SCAL_ALGOS)*len(SCAL_SEEDS)}")
print(f"  λ: freeze={cfg['env']['qos_lambda_freeze_fraction']}, max={cfg['env']['qos_lambda_max']}, "
      f"residual={cfg['env']['phase_residual_scale']}")
print(f"  ⏱️  Ước lượng    ~ {est_min/60.0:.1f} giờ (GPU T4)")
if est_min/60.0 > 11:
    print("  ⚠️  > 11h — KHÔNG vừa 1 session! Dùng RESUME nhiều session (xem cell 0),")
    print("      hoặc giảm N_LIST / SCAL_SEEDS ở trên rồi chạy lại cell này.")
print("=" * 60)

### 4. Huấn luyện lại tại mỗi N (MADDPG vs single-agent)

**📌 Output mong đợi:** với mỗi `N`, tqdm bar cho từng (algo × seed). Mất nhiều giờ — đây là cell nặng nhất.

> Mỗi N: deep-copy config, đặt `num_ris_elements=N`, huấn luyện từ đầu (agent build theo `env.spec()`),
> rồi đánh giá đa-seed với λ huấn luyện-cuối. Kết quả lưu vào `scal_results`.


In [ ]:
# Resume: nạp kết quả đã có (nếu chạy session trước), bỏ qua (N,algo) đã xong.
done = set(); rows = []
if os.path.exists(RESUME_CSV):
    _prev = pd.read_csv(RESUME_CSV)
    rows = _prev.to_dict("records")
    done = {(int(r["N"]), str(r["Algorithm"])) for r in rows}
    print(f"♻️  RESUME: đã có {len(done)} (N,algo): {sorted(done)} — sẽ bỏ qua.\n")

scal_results = {}
for r in rows:   # dựng lại scal_results từ CSV cũ
    scal_results.setdefault(int(r["N"]), {})[str(r["Algorithm"])] = {
        "sr_mean": r["SumRate"], "sr_ci": r["SumRate_CI"],
        "qos_mean": r["QoS"], "qos_ci": r["QoS_CI"]}

for N in N_LIST:
    cfg_N = copy.deepcopy(cfg)
    cfg_N["env"]["num_ris_elements"] = int(N)
    cfg_N["training"]["total_episodes"] = int(SCAL_EPISODES)
    scal_results.setdefault(N, {})
    print(f"\n{'#'*64}\n#  N = {N}  (single-agent action-dim ~ {3*N + 9})\n{'#'*64}")
    for algo in SCAL_ALGOS:
        label = LABEL[algo]
        if (N, label) in done:
            print(f"  -- skip {label} N={N} (đã có trong RESUME)"); continue
        srs, qoss = [], []
        for s in SCAL_SEEDS:
            run = f"scal_{algo}_N{N}_s{s}"
            print(f"\n--- Train {label} | N={N} | seed={s} ---")
            if algo == "maddpg":
                info = train_maddpg(cfg_N, log_dir=log_dir, ckpt_dir=ckpt_dir, seed_override=s, run_name=run)
            else:
                info = train_single_agent(cfg_N, kind=algo, log_dir=log_dir, ckpt_dir=ckpt_dir, seed_override=s, run_name=run)
            lam = info.get("trained_qos_lambda")
            m = _eval_multi_seed(info["agent"], algo, info["obs_norm"], cfg_N,
                                 cfg_N["evaluation"]["seeds"], qos_lambda=lam)
            srs.append(m["sum_rate_mean"]); qoss.append(m["qos_mean"])
        sr_m, sr_ci, _ = confidence_interval(np.array(srs))
        q_m,  q_ci, _  = confidence_interval(np.array(qoss))
        scal_results[N][label] = {"sr_mean": sr_m, "sr_ci": sr_ci, "qos_mean": q_m, "qos_ci": q_ci}
        rows.append({"N": N, "Algorithm": label, "SumRate": sr_m, "SumRate_CI": sr_ci,
                     "QoS": q_m, "QoS_CI": q_ci, "N_seeds": len(SCAL_SEEDS)})
        done.add((N, label))
        # LƯU NGAY sau mỗi (N,algo) -> chống mất dữ liệu nếu session chết
        pd.DataFrame(rows).sort_values(["N", "Algorithm"]).to_csv(CSV_PATH, index=False)
        print(f"  => {label} N={N}: SR={sr_m:.3f}±{sr_ci:.3f}, QoS={q_m:.3f}±{q_ci:.3f}  [đã lưu CSV]")

df_scal = pd.DataFrame(rows).sort_values(["N", "Algorithm"]).reset_index(drop=True)
df_scal.to_csv(CSV_PATH, index=False)
print("\n--- Bảng scalability (scalability_vs_N.csv) ---")
display(df_scal)

### 5. Đồ thị & phân tích lợi thế phân rã

**📌 Output mong đợi:** 3 biểu đồ — (1) SR vs N, (2) QoS vs N, (3) **Decomposition gain vs N**.

**🎯 Bằng chứng cần có:** đường *gain* (MADDPG − best single-agent) **đi lên theo N**.
Nếu đúng ⇒ viết được: *"lợi thế của MADDPG so với đơn-agent tăng theo số phần tử RIS, khẳng định giá trị
của kiến trúc phân rã CTDE khi không gian hành động lớn."*


In [ ]:
labels = [LABEL[a] for a in SCAL_ALGOS]
# Chỉ vẽ các N đã có ĐỦ mọi thuật toán (phòng trường hợp resume chưa xong)
N_plot = [N for N in N_LIST if all(lab in scal_results.get(N, {}) for lab in labels)]
print("N được vẽ:", N_plot)

sr_curves  = {lab: {"mean": [scal_results[N][lab]["sr_mean"] for N in N_plot],
                    "ci":   [scal_results[N][lab]["sr_ci"]   for N in N_plot]} for lab in labels}
plot_metric_vs_x(N_plot, sr_curves, xlabel="STAR-RIS elements $N$",
                 ylabel="Avg. sum-rate (b/s/Hz)", out_dir=fig_dir, name="scal_sumrate_vs_N")

qos_curves = {lab: {"mean": [scal_results[N][lab]["qos_mean"] for N in N_plot],
                    "ci":   [scal_results[N][lab]["qos_ci"]   for N in N_plot]} for lab in labels}
plot_metric_vs_x(N_plot, qos_curves, xlabel="STAR-RIS elements $N$",
                 ylabel="QoS satisfaction probability", out_dir=fig_dir, name="scal_qos_vs_N")

display(Image(filename=os.path.join(fig_dir, "scal_sumrate_vs_N.png")))
display(Image(filename=os.path.join(fig_dir, "scal_qos_vs_N.png")))

if "MADDPG" in labels and len(labels) > 1:
    single = [l for l in labels if l != "MADDPG"]
    gains = [scal_results[N]["MADDPG"]["sr_mean"] - max(scal_results[N][l]["sr_mean"] for l in single)
             for N in N_plot]
    plt.figure(figsize=(5.8, 3.7))
    plt.axhline(0.0, color="gray", lw=0.8, ls="--")
    plt.plot(N_plot, gains, marker="o", color="#1f77b4", lw=2)
    plt.fill_between(N_plot, 0, gains, alpha=0.15, color="#1f77b4")
    plt.xlabel("STAR-RIS elements $N$")
    plt.ylabel("Decomposition gain\nSR(MADDPG) − SR(best single-agent)")
    plt.title("Lợi thế MADDPG tăng theo N")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "scal_decomposition_gain.png"), dpi=150)
    plt.show()
    print("\n--- Decomposition gain theo N ---")
    for N, g in zip(N_plot, gains):
        print(f"  N={N:3d}: gain = {g:+.3f} b/s/Hz   {'✅ MADDPG dẫn' if g>0 else '⚠️ chưa dẫn'}")
    if len(gains) >= 2 and gains[-1] > gains[0]:
        print(f"\n  ✅ XU HƯỚNG ĐÚNG: gain tăng {gains[0]:+.3f} (N={N_plot[0]}) -> {gains[-1]:+.3f} (N={N_plot[-1]})")
        print("     => Củng cố lý thuyết: lợi thế phân rã CTDE tăng theo không gian hành động.")
    else:
        print("\n  ⚠️ Gain chưa tăng đơn điệu — cân nhắc thêm N lớn hơn.")

### 6. Diễn giải cho bài báo

- **Nếu gain tăng theo N** → đây là *đóng góp định lượng* mạnh nhất cho việc chọn MADDPG:
  *"Khi N tăng (không gian hành động RIS phình to), MADDPG vượt dần đơn-agent nhờ phân rã CTDE."*
  Đặt cạnh kết quả latency (MADDPG ~13× nhanh hơn BCD) → câu chuyện hoàn chỉnh: **gần tối ưu, độ trễ thấp,
  và mở rộng tốt theo quy mô RIS.**
- **Nếu gain phẳng/âm** → trung thực: ở dải N này MADDPG ngang đơn-agent; điểm bán là **thực thi phi tập trung
  (decentralized execution)** + latency, không phải sum-rate. Cân nhắc N lớn hơn để lộ xu hướng.
